In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DecimalType, TimestampType
spark = SparkSession.builder.appName("Spark DataFrames").getOrCreate()
df = spark.read.table("dev.bronze.day1_transaction_raw")
pre_schema = StructType([
    StructField("raw_row_id", IntegerType(), False),
    StructField("transaction_id", StringType(), False),
    StructField("customer_id", StringType(), False),
    StructField("merchant_id", StringType(), False),
    StructField("transaction_ts", TimestampType(), True),
    StructField("amount", DecimalType(12, 2), False),
    StructField("status", StringType(), False),
    StructField("source_update_ts", StringType(), True)
])
if df.schema == pre_schema:
    print("Schema Matched")
else:   
    print("Schema Not Matched")

exp_schema = {
    f.name: f.dataType 
    for f in pre_schema.fields
}
act_schema = {
    f.name: f.dataType
    for f in df.schema.fields
}
validation_result = []

for col_value , col_type in exp_schema.items():
    if col_value not in act_schema:
        print(f"missing col, {col_value}")
    elif act_schema[col_value] != col_type:
        print(f"DataType Mismatch : {col_value}, expectedType iS {col_type}, actual is {act_schema[col_value]} ")
    else:
        print(f" {col_value} : {col_type}")


In [0]:
from pyspark.sql.functions import col

df = df.withColumn("source_update_ts", col("source_update_ts").cast(StringType()))
df.printSchema()

In [0]:
from pyspark.sql import SparkSession

from pyspark.sql.types import StructType, StructField, IntegerType, StringType 
spark = SparkSession.builder.appName("Spark df").getOrCreate()

data = [
    (1, "John", "New York",5000),
    (2, "Jane", "Los Angeles",6000),
    (3, "Bob", "Chicago",7000),
    (4, "Alice", "Houston",8000),
    (5, "Charlie", "Miami",9000),
    (6, "David", "Dallas",10000)
]
schema = StructType([
    StructField("id", IntegerType(),True),
    StructField("name", StringType(),True),
    StructField("city", StringType(),True),
    StructField("salary", IntegerType(),True)
])
df = spark.createDataFrame(data, schema = schema)
df.show(5)

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import spark_partition_id, col, length, sum, lit
from functools import reduce

spark = SparkSession.builder.appName("spark init").getOrCreate()
df = spark.read.table("dev.bronze.day1_transaction_raw")
# Get number of partitions using DataFrame API (Spark Connect compatible)
df = df.repartition(5)
num_partitions = df.select(spark_partition_id()).distinct().count()
print(f"Number of partitions: {num_partitions}")
df_filter = df.filter(col("status") == "SUCCESS")
df_filter.count()
# Calculate total character length across all columns
length_cols = [length(col(c).cast("string")) for c in df_filter.columns]
total_length = reduce(lambda x, y: x + y, length_cols, lit(0))
print(total_length)
df_filter.select(sum(total_length)).show()

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = SparkSession.builder.appName("NullsLastExample").getOrCreate()

# Sample data with null values
data = [(1, "Bob"), (2, "Alice"), (3, None), (4, "charlie")]
df = spark.createDataFrame(data, ["id", "name"])

# Sort using the functions module
df_sorted = df.sort(F.desc_nulls_last("name"))
df_sorted.show()


In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.window import Window
from pyspark.sql import functions as f 

dedup_window = Window.partitionBy("transaction_id").orderBy(
    f.col("source_update_ts").desc_nulls_last(),
    f.col("raw_row_id").desc_nulls_last()
)
latest_transaction_df = (
    df.withColumn("version_rank", f.row_number().over(dedup_window)).filter(f.col("version_rank") == 1).drop("version_rank")
)
latest_transaction_df.show()